In [ ]:
# Colab 환경에서 torch, torchvision이 없다면 설치 (대부분은 기본 포함)
!pip install torch torchvision

# helper_functions.py 다운로드 (여기에 download_data 함수가 들어있음)
try:
    from helper_functions import download_data
except ImportError:
    !wget https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py
    from helper_functions import download_data

# 위 명령 결과 출력이 길 수 있으니 %%capture로 숨김.
# 실행이 끝나면 아래 셀로 넘어가세요.

In [ ]:
import torch
import torchvision

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [ ]:
from helper_functions import download_data
image_path = download_data(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
    destination="pizza_steak_sushi"  # 이 폴더명으로 다운로드 & 압축해제
)
print("[INFO] Data downloaded to:", image_path)

[INFO] data/pizza_steak_sushi directory exists, skipping download.
[INFO] Data downloaded to: data/pizza_steak_sushi


In [ ]:
import os
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [ ]:
manual_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dataset = ImageFolder(root=train_dir, transform = manual_transforms)
test_dataset = ImageFolder(root=test_dir,transform = manual_transforms)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size =32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size =32, shuffle =False)

In [ ]:
class_names = train_dataset.classes
print("[INFO] Class Names:", class_names)
print(f"[INFO] Total classes: {len(class_names)}")

[INFO] Class Names: ['pizza', 'steak', 'sushi']
[INFO] Total classes: 3


In [ ]:
import torch.nn as nn

class PatchEmbedding(nn.Module):
  def __init__(self,in_channels:int=3,patch_size:int=16,embedding_dim:int=768):
    super().__init__()
    self.patcher = nn.Conv2d(
        in_channels = in_channels,
        out_channels = embedding_dim,
        kernel_size = patch_size,
        stride = patch_size,
        padding = 0
    )

    self.flatten = nn.Flatten(start_dim=2, end_dim=3)
    self.patch_size = patch_size

  def forward(self,x:torch.Tensor) -> torch.Tensor:
    x = self.patcher(x)
    x = self.flatten(x)
    x = x.permute(0,2,1)
    return x

In [ ]:
class MultiheadSelfAttentionBlock(nn.Module):
  def __init__(self, embedding_dim: int = 768, num_heads: int =12, attn_dropout: float = 0.0):
    super().__init__()

    self.layer_norm = nn.LayerNorm(embedding_dim)

    self.multihead_attn = nn.MultiheadAttention(
        embed_dim = embedding_dim,
        num_heads = num_heads,
        dropout = attn_dropout,
        batch_first = True
        )

  def forward(self, x:torch.Tensor) -> torch.Tensor:
    x_normed = self.layer_norm(x)

    attn_output,  _ = self.multihead_attn(
        x_normed,
        x_normed,
        x_normed,
        need_weights = False
    )
    return attn_output

In [ ]:
class MLPBlock(nn.Module):
  def __init__(self, embedding_dim : int = 768, mlp_size : int = 3072, dropout : float = 0.1):
    super().__init__()
    self.layer_norm = nn.LayerNorm(embedding_dim)

    self.mlp = nn.Sequential(
        nn.Linear(embedding_dim, mlp_size),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(mlp_size, embedding_dim),
        nn.Dropout(dropout)
    )

  def forward(self, x:torch.Tensor) ->torch.Tensor:
    x_normed = self.layer_norm(x)
    x_out = self.mlp(x_normed)
    return x_out

In [ ]:
class TransformerEncoderBlock(nn.Module):
  def __init__(self,embedding_dim : int = 768, num_heads:int =12, mlp_size: int = 3072,mlp_dropout: float = 0.1, attn_dropout: float = 0.0):
    super().__init__()

    self.msa_block = MultiheadSelfAttentionBlock(
        embedding_dim = embedding_dim,
        num_heads = num_heads,
        attn_dropout = attn_dropout
    )

    self.mlp_block = MLPBlock(
        embedding_dim = embedding_dim,
        mlp_size = mlp_size,
        dropout = mlp_dropout
    )
  def forward(self, x:torch.Tensor) -> torch.Tensor:
    x = self.msa_block(x) + x
    x = self.mlp_block(x) + x
    return x

In [ ]:
class ViT(nn.Module):
  def __init__(self, img_size: int =224, in_channels: int =3, patch_size: int = 16,
               num_transformer_layers: int = 12, embedding_dim : int = 768, mlp_size : int =3072,
               num_heads: int = 12, attn_dropout: float = 0.0, mlp_dropout: float = 0.1, embedding_dropout: float = 0.0, num_classes: int = 3):
    super().__init__()

    self.num_patches = (img_size * img_size) // (patch_size ** 2)

    # 클래스 토큰 정의
    self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))

    self.position_embedding = nn.Parameter(torch.randn(1,self.num_patches +1, embedding_dim))

    self.embedding_dropout = nn.Dropout(embedding_dropout)

    self.patch_embedding = PatchEmbedding(
        in_channels = in_channels,
        patch_size = patch_size,
        embedding_dim = embedding_dim
    )

    self.transformer_encoder = nn.Sequential(
        *[
            TransformerEncoderBlock(
                embedding_dim = embedding_dim,
                num_heads = num_heads,
                mlp_size = mlp_size,
                mlp_dropout = mlp_dropout,
                attn_dropout = attn_dropout
            ) for _ in range(num_transformer_layers)
        ]
    )

    self.classifier = nn.Sequential(
        nn.LayerNorm(embedding_dim),
        nn.Linear(embedding_dim, num_classes)
    )
  def forward(self, x: torch.Tensor) -> torch.Tensor:
    batch_size = x.shape[0]
    # class_token = self.class_embedding.expand(batch_size, -1,-1) # 변경 전
    cls_token = self.cls_token.expand(batch_size, -1,-1)

    x = self.patch_embedding(x)
    x = torch.cat((cls_token,x),dim = 1)

    x = x + self.position_embedding
    x = self.embedding_dropout(x)

    x = self.transformer_encoder(x)
    x = self.classifier(x[:,0])

    return x

In [ ]:
42

42

In [ ]:
def train_model(model:nn.Module, train_dataloader:DataLoader, test_dataloader: DataLoader, num_epochs: int = 5, lr: float = 1e-3):
  model.to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),lr = lr)

  for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_dataloader:
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      _, preds = torch.max(outputs, dim = 1)
      train_loss += loss.item() * images.size(0)
      train_correct += torch.sum(preds == labels).item()
      train_total +=  images.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.inference_mode():
      for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        _, preds = torch.max(outputs, dim =1)
        test_loss += loss.item() * images.size(0)
        test_correct += torch.sum(preds == labels).item()
        test_total += images.size(0)

    epoch_test_loss = test_loss / test_total
    epoch_test_acc = test_correct / test_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {epoch_train_loss:.4f}, "
              f"Train Acc: {epoch_train_acc:.4f} | "
              f"Test Loss: {epoch_test_loss:.4f}, "
              f"Test Acc: {epoch_test_acc:.4f}")

In [ ]:
# 클래스 개수는 pizza, steak, sushi => 3개
num_classes = len(class_names)
# ViT 모델 생성
model = ViT(
    num_classes=num_classes  # 분류 대상 = 3
)
# 학습 실행 (5에폭, lr=1e-3)
train_model(
    model=model,
    train_dataloader=train_loader,
    test_dataloader=test_loader,
    num_epochs=42,
    lr=1e-3
)

Epoch [1/42] Train Loss: 3.8200, Train Acc: 0.3422 | Test Loss: 1.1449, Test Acc: 0.4133
Epoch [2/42] Train Loss: 1.2043, Train Acc: 0.3511 | Test Loss: 1.0997, Test Acc: 0.3333
Epoch [3/42] Train Loss: 1.1516, Train Acc: 0.3378 | Test Loss: 1.0819, Test Acc: 0.4133
Epoch [4/42] Train Loss: 1.2150, Train Acc: 0.3467 | Test Loss: 1.2141, Test Acc: 0.2533
Epoch [5/42] Train Loss: 1.2616, Train Acc: 0.3333 | Test Loss: 1.0890, Test Acc: 0.3333
Epoch [6/42] Train Loss: 1.2542, Train Acc: 0.3511 | Test Loss: 1.0949, Test Acc: 0.4133
Epoch [7/42] Train Loss: 1.2786, Train Acc: 0.3511 | Test Loss: 1.3557, Test Acc: 0.3333
Epoch [8/42] Train Loss: 1.3194, Train Acc: 0.3467 | Test Loss: 1.1042, Test Acc: 0.4133
Epoch [9/42] Train Loss: 1.1493, Train Acc: 0.3022 | Test Loss: 1.3147, Test Acc: 0.2533
Epoch [10/42] Train Loss: 1.1873, Train Acc: 0.3289 | Test Loss: 1.0832, Test Acc: 0.4133
Epoch [11/42] Train Loss: 1.1140, Train Acc: 0.3111 | Test Loss: 1.1454, Test Acc: 0.3333
Epoch [12/42] Train